# Study 2: Which design gives useful tradeoffs?

- Notebooks 04-06 suggest that the current reward, mitigation actions, and income may push agents toward frequent high mitigation and little later development.
- I want to understand those parts of the design before trying another set of rules. This notebook starts with the current settings and evidence; proposed changes will be compared one at a time.
- The goal is a useful synthetic experiment, not a claim that one setup is realistic or universally best. This first section does not train agents or save result files.- The selected design keeps nonlinear emissions and prevention-only mitigation, sets high capacity to 20, and combines EDI gain with `(income - base income) / 10` for economic reward. Run Notebook 03 first to create current-design Q-tables; embedded outputs here are historical.

## What would a useful setup show?

- **Environmental challenge:** Development without mitigation can fail, while sensible mitigation can prevent failure. Pollution should still vary enough to compare policies.
- **Meaningful choices:** Low, medium, and high mitigation should have different uses. Paying for high should sometimes do more than medium, and its extra cost should matter.
- **Economic choices:** Development should create a reason to invest over time. Currency should sometimes constrain decisions, rather than accumulate without changing behavior.
- **Fair comparison:** Every proposed setup should be evaluated with the same starts, training settings, and common outcome measures. I will look at pollution, spending, EDI, currency, failures, actions, and variation across training seeds.
- These are criteria for discussing a design, not numerical targets chosen after seeing which reward mode wins.

## Read the current settings

In [57]:
import pandas as pd
import json

from study_2_economic_sustainability.config import EnvironmentConfig
from study_2_economic_sustainability.environment import development_gain
from study_2_economic_sustainability.agents import TabularQLearner
from study_2_economic_sustainability.environment import EconomicSustainabilityEnvironment

config = EnvironmentConfig()

### How much mitigation capacity can an action use?

- The table compares each action's capacity with one region's new emissions at EDI 0 and EDI 100.
- Mitigation only avoids new emissions. Any capacity beyond the gross emissions is unused; it does not remove old pollution. Neighbor spillover is based on the remaining emissions.

In [58]:
action_names = ("save", "develop", "low", "medium", "high")
emissions_at_edi_0 = config.emissions_base
emissions_at_edi_100 = (
    config.emissions_base
    + config.emissions_edi_linear * 100
    + config.emissions_edi_quadratic * 100 ** 2
)

action_rows = []
for action, name in enumerate(action_names):
    capacity = config.mitigation_capacity[action]
    action_rows.append({
        "action": name,
        "cost": config.action_costs[action],
        "capacity": capacity,
        "capacity_after_emissions_at_edi_0": max(0, capacity - emissions_at_edi_0),
        "capacity_after_emissions_at_edi_100": max(0, capacity - emissions_at_edi_100),
    })

display(pd.DataFrame(action_rows).set_index("action"))

,cost,capacity,capacity_after_emissions_at_edi_0,capacity_after_emissions_at_edi_100
action,,,,
save,0,0,0.0,0.0
develop,100,0,0.0,0.0
low,10,5,0.0,0.0
medium,25,15,7.0,0.0
high,50,30,22.0,14.0


- Under the revised formula, gross emissions range from 8 to 28 across EDI 0-100. High capacity is 30, so it can avoid all new emissions at these EDI levels, with at most 2 capacity units unused at EDI 100. Unused capacity does not clean up old pollution.
- The saved table output predates this formula change. Rerun it before using the displayed results to choose action capacities.

### When does income cover a high purchase?

- Income is `20 + 0.5 × EDI` currency per step. A high purchase costs 50.
- `income_minus_high_cost` describes the balance change after one high purchase. The region must have enough currency before receiving that step's income, and it can choose only one action per step.

In [59]:
income_rows = []
for edi in (0, 40, 60, 70, 100):
    income = config.income_base + config.income_per_edi * edi
    income_rows.append({
        "edi": edi,
        "income_per_step": income,
        "income_minus_high_cost": income - config.action_costs[4],
        "edi_gain_if_develop": development_gain(edi),
    })

display(pd.DataFrame(income_rows).set_index("edi"))

,income_per_step,income_minus_high_cost,edi_gain_if_develop
edi,,,
0,20.0,-30.0,10
40,40.0,-10.0,5
60,50.0,0.0,5
70,55.0,5.0,2
100,70.0,20.0,0


- By EDI 60, one step's income covers the full cost of high mitigation. At EDI 70, the region gains 5 currency even after a high purchase. Initial balances are already 100-1,000.
- The EDI gain from a development purchase falls from 10 to 5 and then 2 as EDI rises. Under the selected design, income above the base level also contributes a smaller ongoing reward, so the question is how much that added incentive changes development timing.
- Notebook 05 showed that balances grow after development choices fade. This table shows why affordability alone may provide little reason to invest further.

### How large is the pollution penalty?

- This table shows only the pollution part of local reward at selected resulting loads. It does not predict the full reward of any action.
- Local economic reward combines EDI gain with the ongoing-income bonus. Currency spent on valid actions is still not directly subtracted from reward.

In [60]:
reward_rows = []
for resulting_load in (0, 10, 20, 40, 60, 100):
    pollution_reward = -(resulting_load / 20) ** 2
    reward_rows.append({
        "resulting_pollution_load": resulting_load,
        "pollution_reward": config.pollution_weight * pollution_reward,
    })

display(pd.DataFrame(reward_rows).set_index("resulting_pollution_load"))

,pollution_reward
resulting_pollution_load,
0,-0.00
10,-0.25
20,-1.00
40,-4.00
60,-9.00
100,-25.00


- The pollution penalty is 0 at load 0, -1 at load 20, and -4 at load 40. It grows quickly with load. This gives mitigation a possible reward benefit even below the failure threshold of 100.

## Baseed on the other notebooks though:

- Notebook 04: Original local and mixed agents chose high mitigation on about 85% of region-steps, developed mostly early, and had no failures on the 20 held-out starts.
- Notebook 05: In later stages, medium would have matched 96-97% of paid high actions under local and mixed reward on that single step. High often bought more capacity than was used.
- Notebook 06: Replacing high with medium across whole episodes reduced spending sharply, raised local and mixed GPI modestly, and caused no failures. Training with a 0.02 mitigation-spending reward penalty nearly removed high choices under local and mixed reward, but development choices barely changed. Global results varied more across seeds.

Based on these notebooks and results I have two new questions: 1. When do the agents stop developing and why?  2. Once they have stopped investing in development, what do they do instead?

## When do agents stop developing?

- I'll use the original saved policies, with five training seeds and the same 20 held-out starts as notebooks 04-06.
- For each region in each episode, the last **successful** development action marks the point after which that region never develops again. This is a description of the observed episode, not proof of why the agent stopped.

### Load the original policies

In [61]:
REWARD_MODES = ("local", "global", "mixed")
TRAINING_SEEDS = (11, 23, 37, 41, 53)
EVALUATION_SEEDS = tuple(range(100_000, 100_020))
Q_TABLE_DIRECTORY = "../results/reward_comparison_current"

In [ ]:
agents_by_run = {}
for reward_mode in REWARD_MODES:
    for training_seed in TRAINING_SEEDS:
        filename = f"{Q_TABLE_DIRECTORY}/q_{reward_mode}_{training_seed}.json"
        with open(filename, encoding="utf-8") as file:
            tables = json.load(file)

        agents = []
        for region, table in enumerate(tables):
            agent = TabularQLearner(region)
            for key, values in table.items():
                observation_values = []
                for value in key.split(","):
                    observation_values.append(int(value))
                observation = tuple(observation_values)
                agent.q[observation] = values
            agents.append(agent)
        agents_by_run[(reward_mode, training_seed)] = agents

print(f"Loaded {len(agents_by_run)} policies")


### Record each region's decisions

- `steps_remaining` is the number of steps left under the 200-step limit. The agent does not observe this number, and an episode can end earlier if pollution reaches 100.

In [63]:
def choose_actions(agents, observations):
    actions = []
    for region in range(4):
        actions.append(agents[region].select_action(observations[region], greedy=True))
    return tuple(actions)

In [64]:
def region_rows(environment, actions, info, training_seed, environment_seed):
    rows = []
    for region in range(4):
        rows.append({
            "reward_mode": environment.reward_mode,
            "training_seed": training_seed,
            "environment_seed": environment_seed,
            "region": region,
            "step": environment.step_count,
            "steps_remaining": config.episode_steps - environment.step_count,
            "action": actions[region],
            "valid": info["valid_actions"][region],
            "edi_gain": info["development_gains"][region],
            "edi": info["development_indices"][region],
            "currency": info["currency_balances"][region],
            "regional_pollution": info["pollution_loads"][region],
            "gpi": info["global_pollution_index"],
        })
    return rows

In [65]:
def collect_episode(agents, reward_mode, training_seed, environment_seed):
    environment = EconomicSustainabilityEnvironment(reward_mode)
    observations = environment.reset(environment_seed)
    rows = []

    while True:
        actions = choose_actions(agents, observations)
        observations, rewards, terminated, truncated, info = environment.step(actions)
        rows.extend(region_rows(environment, actions, info, training_seed, environment_seed))
        if terminated or truncated:
            break

    return rows

In [66]:
step_rows = []
for reward_mode in REWARD_MODES:
    for training_seed in TRAINING_SEEDS:
        agents = agents_by_run[(reward_mode, training_seed)]
        for environment_seed in EVALUATION_SEEDS:
            step_rows.extend(collect_episode(
                agents, reward_mode, training_seed, environment_seed
            ))

steps = pd.DataFrame(step_rows)
print(f"Recorded {len(steps):,} region-steps")

Recorded 240,000 region-steps


### Find the last development action

- A successful development action increases EDI. I take the last such action for each region and episode.
- Regions that never develop are counted separately. Regions that end at EDI 100 have reached the development limit, so their later lack of development has a different explanation.

In [67]:
RUN_KEYS = ["reward_mode", "training_seed", "environment_seed", "region"]
last_development = (
    steps[steps["edi_gain"] > 0]
    .sort_values("step")
    .drop_duplicates(RUN_KEYS, keep="last")
    .copy()
)
last_development["at_edi_cap"] = last_development["edi"] >= 100
last_development["can_afford_development"] = (
    (last_development["currency"] >= config.action_costs[1])
    & ~last_development["at_edi_cap"]
)

region_episodes = steps.loc[steps["step"] == 1, RUN_KEYS]
run_status = region_episodes.merge(
    last_development[RUN_KEYS + ["step"]], on=RUN_KEYS, how="left"
)
run_status["never_developed"] = run_status["step"].isna()

In [68]:
never_by_seed = (
    run_status.groupby(["reward_mode", "training_seed"])["never_developed"]
    .mean() * 100
)
never_by_mode = never_by_seed.groupby(level="reward_mode").agg(["mean", "std"])
display(never_by_mode.rename(columns={"mean": "mean_percent", "std": "seed_sd"}).round(1))

,mean_percent,seed_sd
reward_mode,,
global,15.0,22.4
local,0.0,0.0
mixed,0.0,0.0


- This table shows the share of region-episodes with no successful development at all. The mean and standard deviation are across five training seeds.

In [69]:
stop_by_seed = last_development.groupby(["reward_mode", "training_seed"]).agg(
    last_development_step=("step", "mean"),
    steps_remaining=("steps_remaining", "mean"),
    edi=("edi", "mean"),
    currency=("currency", "mean"),
    regional_pollution=("regional_pollution", "mean"),
    gpi=("gpi", "mean"),
    at_edi_cap_percent=("at_edi_cap", "mean"),
    can_afford_development_percent=("can_afford_development", "mean"),
)
stop_by_seed[["at_edi_cap_percent", "can_afford_development_percent"]] *= 100
display(stop_by_seed.groupby(level="reward_mode")[[
    "last_development_step", "steps_remaining", "edi"
]].agg(["mean", "std"]).round(1))

last_development_step      steps_remaining        edi     
                             mean  std            mean  std  mean  std
reward_mode                                                           
global                       29.9  3.4           170.1  3.4  68.9  2.5
local                        25.3  2.5           174.7  2.5  71.0  2.3
mixed                        24.3  2.9           175.7  2.9  68.6  3.0

- `last_development_step` is the final successful development step, averaged within each seed and then across seeds.

In [70]:
display(stop_by_seed.groupby(level="reward_mode")[[
    "currency", "regional_pollution", "gpi",
    "at_edi_cap_percent", "can_afford_development_percent"
]].mean().round(1))

,currency,regional_pollution,gpi,at_edi_cap_percent,can_afford_development_percent
reward_mode,,,,,
global,271.2,27.8,16.8,5.0,95.0
local,189.8,26.9,15.5,3.5,93.8
mixed,211.5,26.3,15.5,0.0,100.0


- These values describe the state just after the last development purchase.
- `can_afford_development_percent` means the region has at least 100 currency for another purchase and its EDI is below 100.
- If many regions stop while they can still afford development, lack of currency alone is unlikely to explain the behavior. High pollution at that point might support an environmental-risk explanation, but the table cannot establish causation.

### What do they choose afterward?

- I count actions from the step after the final successful development purchase until that region's episode ends. Regions that never developed are excluded from this table.
- These are the agents' chosen actions. An unaffordable action is still counted as a choice, so the invalid-action share appears separately.

In [71]:
stops = last_development[RUN_KEYS + ["step"]].rename(
    columns={"step": "last_development_step"}
)
after = steps.merge(stops, on=RUN_KEYS, how="inner")
after = after[after["step"] > after["last_development_step"]].copy()

after_counts = (
    after.groupby(["reward_mode", "training_seed", "action"])
    .size().unstack(fill_value=0).reindex(columns=range(5), fill_value=0)
)
after_percent = after_counts.div(after_counts.sum(axis=1), axis=0) * 100
after_percent.columns = action_names
display(after_percent.groupby(level="reward_mode").mean().round(1))

,save,develop,low,medium,high
reward_mode,,,,,
global,1.5,0.1,0.2,21.1,77.1
local,0.0,0.0,0.0,3.6,96.4
mixed,0.9,0.0,0.0,4.2,94.9


- Each row shows the mean action share across five training seeds after development stops. A large high-mitigation share would confirm what the agents do next, while save and cheaper mitigation shares show the alternatives.

In [72]:
after["invalid"] = ~after["valid"]
invalid_by_seed = (
    after.groupby(["reward_mode", "training_seed"])["invalid"].mean() * 100
)
display(invalid_by_seed.groupby(level="reward_mode").agg(["mean", "std"]).round(1))

,mean,std
reward_mode,,
global,0.1,0.1
local,0.0,0.1
mixed,0.0,0.0


- This is the percentage of post-development choices the environment rejected, averaged across seeds. A high value would mean the chosen-action table needs extra care when describing actual spending.